# Linear trends with a simple correlation coefficient, and a split at the drift onset

Companion to `lat_zones_v6.ipynb`. The data loading and the zone-mean monthly tables (`stats68_n2o` etc.) are built exactly as in that notebook; plots and exploratory cells are left out.

**What is computed**, per gas, pressure level and latitude zone, on the zone-mean monthly series:

- **Full period (2005–2021):** Pearson correlation *r* between mixing ratio and time, the ordinary least-squares (OLS) trend per decade with its 95% CI, the trend in % per decade, and the Theil–Sen slope for comparison with the Mann–Kendall results.
- **Split:** the same trend fitted separately before and after a break year, plus the change in trend and a p-value for that change.
- **Break-year sensitivity:** the before/after trends for several break years, since no exact drift onset is published.

**Choices to know about**

- `DESEASONALIZE = True` removes the mean annual cycle (each calendar month's mean) before fitting. Otherwise *r* mostly measures the size of the seasonal cycle relative to the trend. The trend itself changes little. Set it to `False` to compare directly with the Mann–Kendall cells, which used the raw series.
- **Use `p_adj`, not `p`.** Neighbouring months are not independent, so the plain Pearson p-value is far too small. `p_adj` and the CIs use an effective sample size from the lag-1 autocorrelation of the residuals: n_eff = n(1 − r₁)/(1 + r₁), as in Santer et al. (2000, JGR).
- **Break year:** the MLS data quality documents say the 190-GHz drift started "around 2010", and Livesey et al. (2021) used 2010 as the start of their fitted drift; no exact date is given. The 640-GHz shutdown in mid-2013 is a different event and does not affect the 190-GHz N₂O product used here. 2010 is therefore the default, with 2009–2013 as sensitivity.
- **The "before" segment is short.** It covers 2005–2009, five years or about two QBO cycles, so its trends are uncertain. The useful questions are whether the negative N₂O trends are concentrated after the break, and whether O₃, which is not affected by the N₂O drift, changes in the same way. If N₂O changes and O₃ does not, that points to the instrument rather than the atmosphere.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats as ss
from scipy.stats import linregress, theilslopes
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)

## Data (as in `lat_zones_v6.ipynb`)

In [ ]:
# read files with monthly average values
data_68 = pd.read_csv('monthly_n2o_68.csv')
data_46 = pd.read_csv('monthly_n2o_46.csv')
data_32 = pd.read_csv('monthly_n2o_32.csv')
data_22 = pd.read_csv('monthly_n2o_22.csv')

o3_68 = pd.read_csv('monthly_o3_68.csv')
o3_46 = pd.read_csv('monthly_o3_46.csv')
o3_32 = pd.read_csv('monthly_o3_32.csv')
o3_22 = pd.read_csv('monthly_o3_22.csv')

for df in [data_68, data_46, data_32, data_22, o3_68, o3_46, o3_32, o3_22]:
    df['date'] = pd.to_datetime(df['date'])

In [ ]:
zone_dict = {
    "bashkortostan": 1, "bozeman": 1, "brunei": 2, "california": 1, "catalonia": 1,
    "colombia": 2, "estonia": 1, "finland": 1, "florianopolis": 2, "florida": 2,
    "france": 1, "french_guiana": 2, "huntingdon": 1, "iceland_e": 1, "iceland_w": 1,
    "khabarovsk": 1, "kongo": 2, "kyrgyzstan": 1, "mexico": 2, "morocco": 1,
    "mukhrino": 1, "myanmar": 2, "nz_n": 3, "nz_s": 3, "pantanal": 2,
    "quistococha": 2, "romania": 1, "taiwan": 2, "tarapoto": 2, "tasmania": 3,
    "tierra_del_fuego": 3, "uganda_e": 2, "uganda_n": 2, "uganda_s": 2, "wales": 1
}

data_68["latitude_zone"] = data_68["location"].map(zone_dict)
data_46["latitude_zone"] = data_46["location"].map(zone_dict)
data_32["latitude_zone"] = data_32["location"].map(zone_dict)
data_22["latitude_zone"] = data_22["location"].map(zone_dict)
o3_68["latitude_zone"] = o3_68["location"].map(zone_dict)
o3_46["latitude_zone"] = o3_46["location"].map(zone_dict)
o3_32["latitude_zone"] = o3_32["location"].map(zone_dict)
o3_22["latitude_zone"] = o3_22["location"].map(zone_dict)

data_68 = data_68[data_68['count'] >= 5]
data_46 = data_46[data_46['count'] >= 5]
data_32 = data_32[data_32['count'] >= 5]
data_22 = data_22[data_22['count'] >= 5]
o3_68 = o3_68[o3_68['count'] >= 5]
o3_46 = o3_46[o3_46['count'] >= 5]
o3_32 = o3_32[o3_32['count'] >= 5]
o3_22 = o3_22[o3_22['count'] >= 5]

In [ ]:
def prepare_monthly_stats(
    df: pd.DataFrame,
    date_col: str = 'date',
    zone_col: str = 'latitude_zone',
    value_col: str = 'mean_concentration',
    confidence: float = 0.95
) -> pd.DataFrame:
    """
    Turn a raw DataFrame into monthly, per-zone means ± CI.
    """
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df['year_month'] = df[date_col].dt.to_period('M').dt.to_timestamp()
    stats = (
        df.groupby([zone_col, 'year_month'])[value_col]
          .agg(['mean', 'std', 'count'])
          .reset_index()
    )
    alpha = 1 - confidence
    stats['ci_half'] = stats.apply(
        lambda r: ss.t.ppf(1 - alpha/2, df=r['count']-1) * r['std'] / (r['count']**0.5)
                  if r['count'] > 1 else 0,
        axis=1
    )
    stats['ci_lower'] = stats['mean'] - stats['ci_half']
    stats['ci_upper'] = stats['mean'] + stats['ci_half']
    return stats

stats68_n2o = prepare_monthly_stats(df = data_68)
stats46_n2o = prepare_monthly_stats(df = data_46)
stats32_n2o = prepare_monthly_stats(df = data_32)
stats22_n2o = prepare_monthly_stats(df = data_22)
stats68_o3 = prepare_monthly_stats(df = o3_68)
stats46_o3 = prepare_monthly_stats(df = o3_46)
stats32_o3 = prepare_monthly_stats(df = o3_32)
stats22_o3 = prepare_monthly_stats(df = o3_22)

## Settings and functions
From here on, only the `statsXX_*` tables are used, so these cells can also be pasted into `lat_zones_v6.ipynb` after the cell that creates them.

In [ ]:
DESEASONALIZE = True
YEARS = (2005, 2021)
BREAK = 2010                                # first year of the "after" segment
BREAK_SENSITIVITY = [2009, 2010, 2011, 2012, 2013]

ZONES = {1: "NH", 2: "Tropics", 3: "SH"}
STATS = {
    ("N2O", 68): stats68_n2o, ("N2O", 46): stats46_n2o, ("N2O", 32): stats32_n2o, ("N2O", 22): stats22_n2o,
    ("O3", 68): stats68_o3,   ("O3", 46): stats46_o3,   ("O3", 32): stats32_o3,   ("O3", 22): stats22_o3,
}


def zone_series(stats_df, zone):
    s = (stats_df[stats_df["latitude_zone"] == zone]
         .sort_values("year_month")
         .set_index("year_month")["mean"])
    s.index = pd.DatetimeIndex(s.index)
    return s[(s.index.year >= YEARS[0]) & (s.index.year <= YEARS[1])]


def deseasonalize(s):
    """Subtract each calendar month's mean (the mean annual cycle)."""
    return s - s.groupby(s.index.month).transform("mean")


def decimal_year(idx):
    return np.asarray(idx.year + (idx.month - 0.5) / 12)


def lag1_autocorr(resid):
    """Lag-1 autocorrelation of the residuals, using only consecutive months."""
    full = resid.asfreq("MS")
    a, b = full.values[:-1], full.values[1:]
    ok = ~np.isnan(a) & ~np.isnan(b)
    return np.corrcoef(a[ok], b[ok])[0, 1]


def linear_trend(s, ref_mean):
    """OLS trend of s against time. Pearson r, naive p, and p/CI adjusted for
    lag-1 autocorrelation via an effective sample size. Slopes per decade."""
    t, y = decimal_year(s.index), s.values
    n = len(y)
    fit = linregress(t, y)
    resid = pd.Series(y - (fit.intercept + fit.slope * t), index=s.index)
    r1 = max(lag1_autocorr(resid), 0.0)
    n_eff = max(n * (1 - r1) / (1 + r1), 3.0)
    df = n_eff - 2
    se = fit.stderr * np.sqrt((n - 2) / df)
    half = ss.t.ppf(0.975, df) * se
    return dict(
        n=n, r=fit.rvalue, p=fit.pvalue, p_adj=2 * ss.t.sf(abs(fit.slope / se), df),
        per_decade=10 * fit.slope, ci_low=10 * (fit.slope - half), ci_high=10 * (fit.slope + half),
        pct_per_decade=100 * 10 * fit.slope / ref_mean, lag1=r1, n_eff=n_eff, se_decade=10 * se,
    )


def series_for(gas, lev, zone):
    raw = zone_series(STATS[(gas, lev)], zone)
    return (deseasonalize(raw) if DESEASONALIZE else raw), raw.mean()

## Full period
Slopes are in ppbv (N₂O) or ppmv (O₃) per decade. `sen_per_decade` is the Theil–Sen slope of the same series. With `DESEASONALIZE = False` it matches the Mann–Kendall cells.

In [ ]:
rows = []
for (gas, lev) in STATS:
    for zone, zname in ZONES.items():
        s, ref = series_for(gas, lev, zone)
        res = linear_trend(s, ref)
        res["sen_per_decade"] = 10 * theilslopes(s.values, decimal_year(s.index))[0]
        rows.append(dict(gas=gas, level_hPa=lev, zone=zname, **res))
full = pd.DataFrame(rows)

cols = ["gas", "level_hPa", "zone", "n", "r", "p", "p_adj", "per_decade", "ci_low", "ci_high",
        "pct_per_decade", "sen_per_decade", "lag1", "n_eff"]

## Split at the break year
Separate trends before and after `BREAK`. `change` is after minus before (per decade), and `p_change` tests whether that change differs from zero, using the autocorrelation-adjusted standard errors of both segments.

In [ ]:
def split_row(gas, lev, zone, brk):
    s, ref = series_for(gas, lev, zone)
    b = linear_trend(s[s.index.year < brk], ref)
    a = linear_trend(s[s.index.year >= brk], ref)
    change = a["per_decade"] - b["per_decade"]
    z = change / np.hypot(a["se_decade"], b["se_decade"])
    return dict(
        n_before=b["n"], before=b["per_decade"], before_low=b["ci_low"], before_high=b["ci_high"],
        r_before=b["r"], p_adj_before=b["p_adj"],
        n_after=a["n"], after=a["per_decade"], after_low=a["ci_low"], after_high=a["ci_high"],
        r_after=a["r"], p_adj_after=a["p_adj"],
        change=change, p_change=2 * ss.norm.sf(abs(z)),
    )

split = pd.DataFrame([
    dict(gas=gas, level_hPa=lev, zone=zname, **split_row(gas, lev, zone, BREAK))
    for (gas, lev) in STATS for zone, zname in ZONES.items()
])

## Break-year sensitivity
Trend per decade before and after each candidate break year, for both gases.

In [ ]:
sens = pd.DataFrame([
    dict(gas=gas, level_hPa=lev, zone=zname, break_year=brk,
         **{k: v for k, v in split_row(gas, lev, zone, brk).items() if k in ("before", "after", "p_change")})
    for (gas, lev) in STATS
    for zone, zname in ZONES.items()
    for brk in BREAK_SENSITIVITY
])

## Readable tables
The cells above only compute; these display the results with pandas' `Styler`. Trends are coloured blue (negative) to red (positive) on a scale centred at zero. A trend is **bold** when its autocorrelation-adjusted p is below `SIG`. The full set of columns (naive p, lag-1 autocorrelation, n_eff and so on) stays in the CSV files.

To share a table with its colours, append `.to_excel("name.xlsx")` to any of the display calls (needs `openpyxl`).

In [ ]:
SIG = 0.05           # bold threshold for adjusted p-values
COLOR_LIMIT = None   # trend (per decade) at which colours saturate; None = automatic
UNITS = {"N2O": "ppbv", "O3": "ppmv"}
DECIMALS = {"N2O": 2, "O3": 3}     # decimals shown for trends
BLUE_RED = "RdBu_r"

GROUP_BORDERS = [  # line between pressure levels (3 zones per level)
    {"selector": "tbody tr:nth-child(3n+1) th, tbody tr:nth-child(3n+1) td",
     "props": "border-top: 2px solid #888"},
    {"selector": "caption", "props": "caption-side: top; font-size: 1.1em; font-weight: bold; text-align: left"},
    {"selector": "th, td", "props": "padding: 2px 8px"},
]


def fmt_p(p):
    return "" if pd.isna(p) else ("<0.001" if p < 0.001 else f"{p:.3f}")


def ci_text(lo, hi, dec=2):
    return f"{lo:+.{dec}f} to {hi:+.{dec}f}"


def color_limit(values):
    if COLOR_LIMIT is not None:
        return COLOR_LIMIT
    v = np.abs(np.asarray(values, dtype=float))
    return float(np.nanpercentile(v, 90)) or 1.0


def bold_where(mask):
    return lambda col: np.where(mask.loc[col.index].values, "font-weight: bold", "")


def show_full(gas):
    d = full[full.gas == gas].set_index(["level_hPa", "zone"])
    u, dec = UNITS[gas], DECIMALS[gas]
    num = f"{{:+.{dec}f}}"
    t_col = f"trend ({u}/decade)"
    out = pd.DataFrame({
        "r": d["r"],
        t_col: d["per_decade"],
        "95% CI": [ci_text(a, b, dec) for a, b in zip(d.ci_low, d.ci_high)],
        "trend (%/decade)": d["pct_per_decade"],
        "p (adj.)": d["p_adj"],
        f"Sen slope ({u}/decade)": d["sen_per_decade"],
    })
    lim = color_limit(out[t_col])
    return (out.style
            .format({"r": "{:+.2f}", t_col: num, "trend (%/decade)": "{:+.2f}",
                     f"Sen slope ({u}/decade)": num, "p (adj.)": fmt_p})
            .background_gradient(cmap=BLUE_RED, subset=[t_col], vmin=-lim, vmax=lim)
            .apply(bold_where(out["p (adj.)"] < SIG), subset=[t_col])
            .set_table_styles(GROUP_BORDERS)
            .set_caption(f"{gas}: linear trends {YEARS[0]}–{YEARS[1]} "
                         f"({'deseasonalised' if DESEASONALIZE else 'raw'} zone-mean monthly series)"))


def show_split(gas, brk=BREAK):
    d = split[split.gas == gas].set_index(["level_hPa", "zone"]) if brk == BREAK else \
        pd.DataFrame([dict(level_hPa=lev, zone=zname, **split_row(gas, lev, zone, brk))
                      for (g, lev) in STATS if g == gas for zone, zname in ZONES.items()]
                     ).set_index(["level_hPa", "zone"])
    u, dec = UNITS[gas], DECIMALS[gas]
    b, a = f"{YEARS[0]}–{brk - 1}", f"{brk}–{YEARS[1]}"
    out = pd.DataFrame({
        (b, "trend"): d["before"], (b, "95% CI"): [ci_text(x, y, dec) for x, y in zip(d.before_low, d.before_high)],
        (b, "p"): d["p_adj_before"],
        (a, "trend"): d["after"], (a, "95% CI"): [ci_text(x, y, dec) for x, y in zip(d.after_low, d.after_high)],
        (a, "p"): d["p_adj_after"],
        ("change", "after − before"): d["change"], ("change", "p"): d["p_change"],
    })
    out.columns = pd.MultiIndex.from_tuples(out.columns)
    trend_cols = [(b, "trend"), (a, "trend"), ("change", "after − before")]
    lim = color_limit(out[trend_cols].values)
    formats = {c: f"{{:+.{dec}f}}" for c in trend_cols}
    formats.update({c: fmt_p for c in [(b, "p"), (a, "p"), ("change", "p")]})
    st = (out.style
          .format(formats)
          .background_gradient(cmap=BLUE_RED, subset=trend_cols, vmin=-lim, vmax=lim, axis=None))
    for t_c, p_c in [((b, "trend"), (b, "p")), ((a, "trend"), (a, "p")),
                     (("change", "after − before"), ("change", "p"))]:
        st = st.apply(bold_where(out[p_c] < SIG), subset=[t_c])
    return (st.set_table_styles(GROUP_BORDERS)
              .set_table_styles({(a, "trend"): [{"selector": "td", "props": "border-left: 2px solid #888"}],
                                 ("change", "after − before"): [{"selector": "td", "props": "border-left: 2px solid #888"}]},
                                overwrite=False)
              .set_caption(f"{gas}: trends ({u}/decade) before and after {brk}"))


def show_sensitivity(gas):
    d = sens[sens.gas == gas]
    u = UNITS[gas]
    dec = DECIMALS[gas] - 1
    wide = d.pivot_table(index=["level_hPa", "zone"], columns="break_year",
                         values=["before", "after", "p_change"], sort=False)
    years = sorted(d.break_year.unique())
    out = pd.DataFrame({
        **{("before the break", f"{YEARS[0]}–{y - 1}"): wide[("before", y)] for y in years},
        **{("after the break", f"{y}–{YEARS[1]}"): wide[("after", y)] for y in years},
    })
    out.columns = pd.MultiIndex.from_tuples(out.columns)
    lim = color_limit(out.values)
    sig = pd.DataFrame({("after the break", f"{y}–{YEARS[1]}"): wide[("p_change", y)] < SIG for y in years})
    st = (out.style
          .format(f"{{:+.{dec}f}}")
          .background_gradient(cmap=BLUE_RED, vmin=-lim, vmax=lim, axis=None))
    for col in sig.columns:
        st = st.apply(bold_where(sig[col]), subset=[col])
    first_after = ("after the break", f"{years[0]}–{YEARS[1]}")
    return (st.set_table_styles(GROUP_BORDERS)
              .set_table_styles({first_after: [{"selector": "td", "props": "border-left: 2px solid #888"}]},
                                overwrite=False)
              .set_caption(f"{gas}: trend ({u}/decade) for each split; column headers give the years fitted. "
                           f"Bold 'after' values: change after − before has p < {SIG}."))

### Full period

In [ ]:
show_full("N2O")

In [ ]:
show_full("O3")

### Split at `BREAK`
For another break year, pass it directly, e.g. `show_split("N2O", 2011)`.

In [ ]:
show_split("N2O")

In [ ]:
show_split("O3")

### Break-year sensitivity

In [ ]:
show_sensitivity("N2O")

In [ ]:
show_sensitivity("O3")

## Mann–Kendall with corrections for seasonality and autocorrelation

`mk.original_test`, as used in `lat_zones_v6.ipynb` and Table S1, assumes every month is independent of the next. In these series they are not, so its p-values are too small. Two standard corrections from the same package:

- **Hamed–Rao** (Hamed and Rao, 1998, *J. Hydrol.* 204, 182–196) on the **deseasonalised** series: the Mann–Kendall statistic with its variance inflated according to the autocorrelation in the data. This is the main suggestion, since it parallels the linear analysis above (deseasonalised series, autocorrelation-adjusted p). τ and the Sen slope come from this run.
- **Seasonal Kendall with the Hirsch–Slack correction** (Hirsch et al., 1982, *Water Resour. Res.* 18, 107–121; Hirsch and Slack, 1984, *Water Resour. Res.* 20, 727–732) on the **raw** series: trends are computed within each calendar month (January against January, and so on) and combined, with a correction for correlation between months. It needs no deseasonalising, so it serves as a cross-check.

`p original` reproduces the existing test for comparison. The last column flags where a trend counted as significant before but not after the Hamed–Rao correction.

Neither correction does anything about the instrument drift. They make the p-values honest, nothing more.

In [ ]:
import pymannkendall as mk

mk_rows = []
for (gas, lev) in STATS:
    for zone, zname in ZONES.items():
        raw = zone_series(STATS[(gas, lev)], zone).asfreq("MS")   # NaN for any missing month
        des = deseasonalize(raw)
        orig = mk.original_test(raw.values)                       # as in lat_zones_v6 / Table S1
        hr = mk.hamed_rao_modification_test(des.values)          # monthly steps -> x120 per decade
        sk = mk.correlated_seasonal_test(raw.values, period=12)  # yearly steps  -> x10 per decade
        mk_rows.append(dict(
            gas=gas, level_hPa=lev, zone=zname,
            tau=hr.Tau, sen_per_decade=120 * hr.slope, pct_per_decade=100 * 120 * hr.slope / raw.mean(),
            p_original=orig.p, p_hamed_rao=hr.p,
            sen_seasonal_per_decade=10 * sk.slope, p_seasonal_kendall=sk.p,
        ))
mk_table = pd.DataFrame(mk_rows)

In [ ]:
def show_mk(gas):
    d = mk_table[mk_table.gas == gas].set_index(["level_hPa", "zone"])
    u, dec = UNITS[gas], DECIMALS[gas]
    num = f"{{:+.{dec}f}}"
    s_col, ss_col = f"Sen slope ({u}/decade)", f"seasonal Sen slope ({u}/decade)"
    flag = np.where(d.p_original < SIG, "yes", "no").astype(object) + " → " + \
           np.where(d.p_hamed_rao < SIG, "yes", "no").astype(object)
    out = pd.DataFrame({
        "τ": d["tau"], s_col: d["sen_per_decade"], "%/decade": d["pct_per_decade"],
        "p original": d["p_original"], "p Hamed–Rao": d["p_hamed_rao"],
        ss_col: d["sen_seasonal_per_decade"], "p seasonal Kendall": d["p_seasonal_kendall"],
        f"significant (p < {SIG}): original → Hamed–Rao": flag,
    }, index=d.index)
    lim = color_limit(out[s_col])
    p_cols = ["p original", "p Hamed–Rao", "p seasonal Kendall"]
    st = (out.style
          .format({"τ": "{:+.3f}", s_col: num, "%/decade": "{:+.2f}", ss_col: num, **{c: fmt_p for c in p_cols}})
          .background_gradient(cmap=BLUE_RED, subset=[s_col], vmin=-lim, vmax=lim))
    for c in p_cols:
        st = st.apply(bold_where(out[c] < SIG), subset=[c])
    lost = out.columns[-1]
    st = st.apply(lambda col: np.where(col == "yes → no", "background-color: #fde3b0; font-weight: bold", ""),
                  subset=[lost])
    return (st.set_table_styles(GROUP_BORDERS)
              .set_caption(f"{gas}: Mann–Kendall trends {YEARS[0]}–{YEARS[1]}, original and corrected"))

In [ ]:
show_mk("N2O")

In [ ]:
show_mk("O3")

## Save

In [ ]:
full[cols].to_csv("trends_linear_full.csv", sep=";", index=False)
split.to_csv(f"trends_linear_split_{BREAK}.csv", sep=";", index=False)
sens.to_csv("trends_break_sensitivity.csv", sep=";", index=False)
mk_table.to_csv("trends_mann_kendall_corrected.csv", sep=";", index=False)

## Mann–Kendall before and after the break

The same two corrected tests as above, run separately on the years before and after `BREAK` (2005–2009 and 2010–2021 by default):

- **Hamed–Rao** on the deseasonalised series, removing each calendar month's 2005–2021 mean as in the linear split. τ and the Sen slope come from this test.
- **Seasonal Kendall** with the Hirsch–Slack correction, on the raw series.

A Sen slope is **bold** when both tests give p < `SIG`, the rule used for Table S1.

**Caveat for the short segment.** Before 2010 the seasonal Kendall test has only five values per calendar month, so its p-values there are crude, and it rarely reaches significance. Mann–Kendall has no standard test for whether the trend *changed*. For that, use `p_change` in the linear split table above. The `change` column here is simply after minus before.

For another break year, pass it directly, e.g. `show_mk_split("N2O", 2011)`. The break has to be a whole year so that the seasonal test's months line up.

In [ ]:
def mk_segment(raw_seg, des_seg, ref_mean):
    hr = mk.hamed_rao_modification_test(des_seg.values)          # monthly steps -> x120 per decade
    sk = mk.correlated_seasonal_test(raw_seg.values, period=12)  # yearly steps  -> x10 per decade
    return dict(n=int(des_seg.notna().sum()), tau=hr.Tau, sen=120 * hr.slope,
                pct=100 * 120 * hr.slope / ref_mean, p_hr=hr.p, sen_sk=10 * sk.slope, p_sk=sk.p)


def mk_split_table(brk=BREAK):
    rows = []
    for (gas, lev) in STATS:
        for zone, zname in ZONES.items():
            raw = zone_series(STATS[(gas, lev)], zone).asfreq("MS")
            des = deseasonalize(raw)                     # full-period monthly means, as above
            row = dict(gas=gas, level_hPa=lev, zone=zname, break_year=brk)
            for seg, sel in (("before", raw.index.year < brk), ("after", raw.index.year >= brk)):
                res = mk_segment(raw[sel], des[sel], raw.mean())
                row.update({f"{seg}_{k}": v for k, v in res.items()})
            row["change_sen"] = row["after_sen"] - row["before_sen"]
            rows.append(row)
    return pd.DataFrame(rows)


mk_split = mk_split_table(BREAK)

In [ ]:
def show_mk_split(gas, brk=BREAK):
    d = mk_split if brk == BREAK else mk_split_table(brk)
    d = d[d.gas == gas].set_index(["level_hPa", "zone"])
    u, dec = UNITS[gas], DECIMALS[gas]
    num = f"{{:+.{dec}f}}"
    b, a = f"{YEARS[0]}–{brk - 1}", f"{brk}–{YEARS[1]}"
    s_lab = f"Sen slope ({u}/decade)"
    out = pd.DataFrame({
        (b, s_lab): d.before_sen, (b, "τ"): d.before_tau,
        (b, "p Hamed–Rao"): d.before_p_hr, (b, "p seasonal Kendall"): d.before_p_sk,
        (a, s_lab): d.after_sen, (a, "τ"): d.after_tau,
        (a, "p Hamed–Rao"): d.after_p_hr, (a, "p seasonal Kendall"): d.after_p_sk,
        ("change", "after − before"): d.change_sen,
    })
    out.columns = pd.MultiIndex.from_tuples(out.columns)
    slope_cols = [(b, s_lab), (a, s_lab), ("change", "after − before")]
    p_cols = [(seg, t) for seg in (b, a) for t in ("p Hamed–Rao", "p seasonal Kendall")]
    lim = color_limit(out[slope_cols].values)
    formats = {c: num for c in slope_cols}
    formats.update({(seg, "τ"): "{:+.3f}" for seg in (b, a)})
    formats.update({c: fmt_p for c in p_cols})
    st = (out.style.format(formats)
          .background_gradient(cmap=BLUE_RED, subset=slope_cols, vmin=-lim, vmax=lim, axis=None))
    for seg in (b, a):
        both = (out[(seg, "p Hamed–Rao")] < SIG) & (out[(seg, "p seasonal Kendall")] < SIG)
        st = st.apply(bold_where(both), subset=[(seg, s_lab)])
        for c in ("p Hamed–Rao", "p seasonal Kendall"):
            st = st.apply(bold_where(out[(seg, c)] < SIG), subset=[(seg, c)])
    return (st.set_table_styles(GROUP_BORDERS)
              .set_table_styles({(a, s_lab): [{"selector": "td", "props": "border-left: 2px solid #888"}],
                                 ("change", "after − before"): [{"selector": "td", "props": "border-left: 2px solid #888"}]},
                                overwrite=False)
              .set_caption(f"{gas}: Mann–Kendall trends before and after {brk}. "
                           f"Bold Sen slope: p < {SIG} in both corrected tests."))

In [ ]:
show_mk_split("N2O")

In [ ]:
show_mk_split("O3")

In [ ]:
mk_split.to_csv(f"trends_mann_kendall_split_{BREAK}.csv", sep=";", index=False)

## Combined trends cell

In [ ]:
def show_trends_combined(gas):
    """Full-period trends side by side: corrected Mann–Kendall (Sen slope, τ, Hamed–Rao and
    seasonal Kendall p) and the linear analysis (OLS trend with 95% CI, Pearson r,
    autocorrelation-adjusted p). Sen slopes are bold when both Mann–Kendall p < SIG
    (the Table S1 rule); linear trends are bold when the adjusted p < SIG."""
    m = mk_table[mk_table.gas == gas].set_index(["level_hPa", "zone"])
    f = full[full.gas == gas].set_index(["level_hPa", "zone"]).loc[m.index]
    u, dec = UNITS[gas], DECIMALS[gas]
    num = f"{{:+.{dec}f}}"
    MK, LIN = "Mann–Kendall", "Linear"
    sen, ols = (MK, f"Sen slope ({u}/decade)"), (LIN, f"trend ({u}/decade)")
    out = pd.DataFrame({
        sen: m["sen_per_decade"],
        (MK, "%/decade"): m["pct_per_decade"],
        (MK, "τ"): m["tau"],
        (MK, "p Hamed–Rao"): m["p_hamed_rao"],
        (MK, "p seasonal Kendall"): m["p_seasonal_kendall"],
        ols: f["per_decade"],
        (LIN, "95% CI"): [ci_text(a, b, dec) for a, b in zip(f.ci_low, f.ci_high)],
        (LIN, "r"): f["r"],
        (LIN, "p (adj.)"): f["p_adj"],
    }, index=m.index)
    out.columns = pd.MultiIndex.from_tuples(out.columns)
    p_cols = [(MK, "p Hamed–Rao"), (MK, "p seasonal Kendall"), (LIN, "p (adj.)")]
    lim = color_limit(out[[sen, ols]].values)
    st = (out.style
          .format({sen: num, ols: num, (MK, "%/decade"): "{:+.2f}", (MK, "τ"): "{:+.3f}",
                   (LIN, "r"): "{:+.2f}", **{c: fmt_p for c in p_cols}})
          .background_gradient(cmap=BLUE_RED, subset=[sen, ols], vmin=-lim, vmax=lim, axis=None)
          .apply(bold_where((out[p_cols[0]] < SIG) & (out[p_cols[1]] < SIG)), subset=[sen])
          .apply(bold_where(out[p_cols[2]] < SIG), subset=[ols]))
    return (st.set_table_styles(GROUP_BORDERS)
              .set_table_styles({ols: [{"selector": "td", "props": "border-left: 2px solid #888"}]}, overwrite=False)
              .set_caption(f"{gas}: trends {YEARS[0]}–{YEARS[1]}. Bold Sen slope: p < {SIG} in both "
                           f"Mann–Kendall tests; bold linear trend: adjusted p < {SIG}."))

In [ ]:
show_trends_combined("N2O")

In [ ]:
show_trends_combined("O3")

In [ ]:
# Supplement tables: versions of show_split / show_sensitivity / show_trends_combined whose
# vertical separator lines are cell styles, so they also reach Excel (table_styles are HTML-only).
# Paste this cell after the show_mk cell; it replaces the earlier show_split and show_sensitivity.

INDEX_NAMES = ["Level (hPa)", "Zone"]   # row labels shown in the tables

def vline(col):
    """Left border on every cell of one column."""
    return lambda c: ["border-left: 2px solid #888"] * len(c)


def show_split(gas, brk=BREAK, ci=True):
    """Trends before and after brk; ci=False leaves out the 95% CI columns."""
    d = split[split.gas == gas].set_index(["level_hPa", "zone"]) if brk == BREAK else \
        pd.DataFrame([dict(level_hPa=lev, zone=zname, **split_row(gas, lev, zone, brk))
                      for (g, lev) in STATS if g == gas for zone, zname in ZONES.items()]
                     ).set_index(["level_hPa", "zone"])
    u, dec = UNITS[gas], DECIMALS[gas]
    b, a = f"{YEARS[0]}–{brk - 1}", f"{brk}–{YEARS[1]}"
    cols = {(b, "trend"): d["before"]}
    if ci:
        cols[(b, "95% CI")] = [ci_text(x, y, dec) for x, y in zip(d.before_low, d.before_high)]
    cols[(b, "p")] = d["p_adj_before"]
    cols[(a, "trend")] = d["after"]
    if ci:
        cols[(a, "95% CI")] = [ci_text(x, y, dec) for x, y in zip(d.after_low, d.after_high)]
    cols[(a, "p")] = d["p_adj_after"]
    cols[("change", "after − before")] = d["change"]
    cols[("change", "p")] = d["p_change"]
    out = pd.DataFrame(cols)
    out.columns = pd.MultiIndex.from_tuples(out.columns)
    out.index.names = INDEX_NAMES
    trend_cols = [(b, "trend"), (a, "trend"), ("change", "after − before")]
    p_cols = [(b, "p"), (a, "p"), ("change", "p")]
    lim = color_limit(out[trend_cols].values)
    st = (out.style
          .format({**{c: f"{{:+.{dec}f}}" for c in trend_cols}, **{c: fmt_p for c in p_cols}})
          .background_gradient(cmap=BLUE_RED, subset=trend_cols, vmin=-lim, vmax=lim, axis=None))
    for t_c, p_c in zip(trend_cols, p_cols):
        st = st.apply(bold_where(out[p_c] < SIG), subset=[t_c])
    for c in [(a, "trend"), ("change", "after − before")]:
        st = st.apply(vline(c), subset=[c])
    return (st.set_table_styles(GROUP_BORDERS)
              .set_caption(f"{gas}: trends ({u}/decade) before and after {brk}. "
                           f"Bold: autocorrelation-adjusted p < {SIG}."))


def show_sensitivity(gas):
    d = sens[sens.gas == gas]
    u = UNITS[gas]
    dec = DECIMALS[gas] - 1
    wide = d.pivot_table(index=["level_hPa", "zone"], columns="break_year",
                         values=["before", "after", "p_change"], sort=False)
    years = sorted(d.break_year.unique())
    out = pd.DataFrame({
        **{("before the break", f"{YEARS[0]}–{y - 1}"): wide[("before", y)] for y in years},
        **{("after the break", f"{y}–{YEARS[1]}"): wide[("after", y)] for y in years},
    })
    out.columns = pd.MultiIndex.from_tuples(out.columns)
    out.index.names = INDEX_NAMES
    lim = color_limit(out.values)
    sig = pd.DataFrame({("after the break", f"{y}–{YEARS[1]}"): wide[("p_change", y)] < SIG for y in years})
    st = (out.style
          .format(f"{{:+.{dec}f}}")
          .background_gradient(cmap=BLUE_RED, vmin=-lim, vmax=lim, axis=None))
    for col in sig.columns:
        st = st.apply(bold_where(sig[col]), subset=[col])
    first_after = ("after the break", f"{years[0]}–{YEARS[1]}")
    st = st.apply(vline(first_after), subset=[first_after])
    return (st.set_table_styles(GROUP_BORDERS)
              .set_caption(f"{gas}: trend ({u}/decade) for each split; column headers give the years fitted. "
                           f"Bold 'after' values: change after − before has p < {SIG}."))


def show_trends_combined(gas):
    """Full-period trends side by side: corrected Mann–Kendall (Sen slope, τ, Hamed–Rao and
    seasonal Kendall p) and the linear analysis (OLS trend with 95% CI, Pearson r,
    autocorrelation-adjusted p). Sen slopes are bold when both Mann–Kendall p < SIG
    (the Table S1 rule); linear trends are bold when the adjusted p < SIG."""
    m = mk_table[mk_table.gas == gas].set_index(["level_hPa", "zone"])
    f = full[full.gas == gas].set_index(["level_hPa", "zone"]).loc[m.index]
    u, dec = UNITS[gas], DECIMALS[gas]
    num = f"{{:+.{dec}f}}"
    MK, LIN = "Mann–Kendall", "Linear"
    sen, ols = (MK, f"Sen slope ({u}/decade)"), (LIN, f"trend ({u}/decade)")
    out = pd.DataFrame({
        sen: m["sen_per_decade"],
        (MK, "%/decade"): m["pct_per_decade"],
        (MK, "τ"): m["tau"],
        (MK, "p Hamed–Rao"): m["p_hamed_rao"],
        (MK, "p seasonal Kendall"): m["p_seasonal_kendall"],
        ols: f["per_decade"],
        (LIN, "95% CI"): [ci_text(a, b, dec) for a, b in zip(f.ci_low, f.ci_high)],
        (LIN, "r"): f["r"],
        (LIN, "p (adj.)"): f["p_adj"],
    }, index=m.index)
    out.columns = pd.MultiIndex.from_tuples(out.columns)
    out.index.names = INDEX_NAMES
    p_cols = [(MK, "p Hamed–Rao"), (MK, "p seasonal Kendall"), (LIN, "p (adj.)")]
    lim = color_limit(out[[sen, ols]].values)
    st = (out.style
          .format({sen: num, ols: num, (MK, "%/decade"): "{:+.2f}", (MK, "τ"): "{:+.3f}",
                   (LIN, "r"): "{:+.2f}", **{c: fmt_p for c in p_cols}})
          .background_gradient(cmap=BLUE_RED, subset=[sen, ols], vmin=-lim, vmax=lim, axis=None)
          .apply(bold_where((out[p_cols[0]] < SIG) & (out[p_cols[1]] < SIG)), subset=[sen])
          .apply(bold_where(out[p_cols[2]] < SIG), subset=[ols])
          .apply(vline(ols), subset=[ols]))
    return (st.set_table_styles(GROUP_BORDERS)
              .set_caption(f"{gas}: trends {YEARS[0]}–{YEARS[1]}. Bold Sen slope: p < {SIG} in both "
                           f"Mann–Kendall tests; bold linear trend: adjusted p < {SIG}."))

In [ ]:
from styled_excel_export import save_styled_excel
save_styled_excel({
    "N2O trends": show_trends_combined("N2O"),
    "O3 trends": show_trends_combined("O3"),
    "N2O split 2010": show_split("N2O"),
    "O3 split 2010": show_split("O3"),
    "N2O break sensitivity": show_sensitivity("N2O"),
}, "supplement_trend_tables.xlsx")